# Statistics & EDA Lab: Understanding Your Data Before Modeling

## Student Practice Notebook

**Name:** Preetham Manchikanti 
**Register Number:** AP24110011315  
**Date:** 28/08/26  

## Learning Objectives

After completing this lab, you will be able to:

- Compute and interpret descriptive statistics beyond `.describe()` (variance, std, skewness)
- Visualize and identify distribution shapes (normal vs skewed)
- Compute and interpret correlation between features
- Run a basic hypothesis test (t-test) and interpret a p-value
- Understand confidence intervals conceptually
- Apply statistical thinking to image and text data

### Why this week matters
You now have clean, scaled, encoded data. Before building any model, a good data scientist asks: **what does this data actually look like? Which features seem related? Is a difference I'm seeing real, or just noise?** That's what statistics answers.

### How to use this notebook
Same as before: **demo → practice → predict-then-run → reflect.**


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.precision', 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips.head()


,total_bill,tip,sex,smoker,day,time,size,tip_pct
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059
1,10.34,1.66,Male,No,Sun,Dinner,3,0.161
2,21.01,3.50,Male,No,Sun,Dinner,3,0.167
3,23.68,3.31,Male,No,Sun,Dinner,2,0.140
4,24.59,3.61,Female,No,Sun,Dinner,4,0.147


## Part 1: Descriptive Statistics Beyond `.describe()`

`.describe()` gives mean, std, min/max, quartiles. Two more measures matter a lot for understanding data shape:

- **Variance / Std Dev**: how spread out the data is
- **Skewness**: whether the data leans left (negative skew) or right (positive skew), or is symmetric (skew ≈ 0)

### Demonstration


In [ ]:
print("Mean:", tips['total_bill'].mean())
print("Median:", tips['total_bill'].median())
print("Variance:", tips['total_bill'].var())
print("Std Dev:", tips['total_bill'].std())
print("Skewness:", tips['total_bill'].skew())


**Reading skewness:**
- Skew ≈ 0 → roughly symmetric
- Skew > 0 → right-skewed (long tail of high values; mean > median)
- Skew < 0 → left-skewed (long tail of low values; mean < median)


### Student Practice

Compute mean, median, variance, std dev, and skewness for the `tip_pct` column. Based on the mean vs median relationship, is it left-skewed, right-skewed, or roughly symmetric? Write your conclusion.


In [ ]:
# Write your answer here


**Reflection:** Why can two columns have the *same mean* but very different std dev / skewness? Why does this matter when deciding how to preprocess a feature (recall Week 3 scaling/outliers)?

*Your answer:*


## Part 2: Visualizing Distributions

Numbers alone can be misleading — always look at the shape too.

### Demonstration


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))

axes[0].hist(tips['total_bill'], bins=20, edgecolor='black')
axes[0].set_title("Total Bill Distribution")
axes[0].set_xlabel("Total Bill")

axes[1].boxplot(tips['total_bill'])
axes[1].set_title("Total Bill Boxplot")

plt.tight_layout()
plt.show()


### Student Practice — predict then check

Predict: will `tip_pct`'s histogram look more symmetric (bell-shaped) or skewed, based on your Part 1 answer? Write your prediction, then plot a histogram and boxplot for `tip_pct` to check.

*Your prediction:*


In [ ]:
# Write your answer here


## Part 3: Correlation

Correlation measures how strongly two numeric variables move together, from -1 (perfectly inverse) to +1 (perfectly aligned). 0 means no linear relationship.

### Demonstration


In [ ]:
correlation = tips['total_bill'].corr(tips['tip'])
print(f"Correlation between total_bill and tip: {correlation:.3f}")

plt.figure()
plt.scatter(tips['total_bill'], tips['tip'], alpha=0.6)
plt.title("Total Bill vs Tip")
plt.xlabel("Total Bill")
plt.ylabel("Tip")
plt.show()


In [ ]:
# Correlation matrix across all numeric columns
numeric_cols = tips.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()
corr_matrix


In [ ]:
# Visualize as a heatmap
plt.figure()
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


### Student Practice

1. What is the correlation between `size` (party size) and `total_bill`? Is it weak, moderate, or strong?
2. What is the correlation between `size` and `tip_pct`? Compare this to `size` vs `total_bill` — which relationship is stronger, and does that surprise you?


In [ ]:
# Write your answer here


**Reflection:** Correlation measures a *linear* relationship. Can two variables be strongly related but have a correlation near 0? Explain how that's possible (hint: think about non-linear/curved relationships).

*Your answer:*


## Part 4: Hypothesis Testing — Is a Difference Real?

You've seen group averages differ (e.g. average tip_pct by day). But is that difference **statistically significant**, or could it just be random variation? A **t-test** compares two group means and gives a **p-value**: the probability of seeing a difference this large (or larger) if there were truly no difference.

**Rule of thumb:** if p-value < 0.05, we consider the difference statistically significant.

### Demonstration: do smokers tip a different percentage than non-smokers?


In [ ]:
smoker_tips = tips[tips['smoker']=='Yes']['tip_pct']
nonsmoker_tips = tips[tips['smoker']=='No']['tip_pct']

print("Smoker mean tip_pct:", smoker_tips.mean())
print("Non-smoker mean tip_pct:", nonsmoker_tips.mean())

t_stat, p_value = stats.ttest_ind(smoker_tips, nonsmoker_tips)
print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.3f}")

if p_value < 0.05:
    print("Result: statistically significant difference")
else:
    print("Result: NOT a statistically significant difference (could be random variation)")


### Student Practice — predict then check

Predict: do you think there's a statistically significant difference in `tip_pct` between Lunch and Dinner? Write your prediction and reasoning, then run a t-test to check.

*Your prediction:*


In [ ]:
# Write your answer here


**Reflection:** Why is it important to check statistical significance instead of just comparing two mean values directly and assuming any difference is meaningful?

*Your answer:*


## Part 5: Confidence Intervals (Conceptual Introduction)

A **confidence interval (CI)** gives a range where we're reasonably confident the true population mean lies, based on our sample. A 95% CI means: if we repeated this sampling process many times, about 95% of such intervals would contain the true mean.

### Demonstration


In [ ]:
mean_bill = tips['total_bill'].mean()
sem = stats.sem(tips['total_bill'])  # standard error of the mean
ci = stats.t.interval(0.95, len(tips)-1, loc=mean_bill, scale=sem)

print(f"Sample mean total_bill: {mean_bill:.2f}")
print(f"95% Confidence Interval: ({ci[0]:.2f}, {ci[1]:.2f})")


### Student Practice

Compute the 95% confidence interval for `tip_pct`. In one sentence, explain what this interval tells you (in plain language, not just formula terms).


In [ ]:
# Write your answer here


# Mini Project A: Statistical Comparison of Images (Image Track)

Using the image metadata DataFrame you built in Week 2 (filename, brightness, RGB averages, etc.), apply statistical thinking to compare your images.

## Step 1: Rebuild your image metadata table


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from PIL import Image
import numpy as np

filenames = list(uploaded.keys())
rows = []
for name in filenames:
    arr = np.array(Image.open(name))
    rows.append({
        'filename': name,
        'avg_brightness': np.mean(arr),
        'avg_red': np.mean(arr[:,:,0]) if arr.ndim==3 else np.mean(arr),
        'avg_green': np.mean(arr[:,:,1]) if arr.ndim==3 else np.mean(arr),
        'avg_blue': np.mean(arr[:,:,2]) if arr.ndim==3 else np.mean(arr),
        'height': arr.shape[0],
        'width': arr.shape[1],
    })

image_df = pd.DataFrame(rows)
image_df


## Step 2: Statistical analysis

1. Compute the mean, std dev, and skewness of `avg_brightness` across your images.
2. Compute the correlation between `avg_brightness` and `avg_red` (do brighter images tend to have more red?).
3. **If you have at least 2 images you can meaningfully group** (e.g. indoor vs outdoor, or any 2 categories you choose), run a t-test comparing their `avg_brightness`. If you only have very few images, skip the t-test and instead just discuss whether a t-test would even be reliable with so few samples.


In [ ]:
# Write your answer here


**Reflection:** With only 3-5 images, can you trust statistical significance results (like a t-test p-value) the same way you would with 200+ rows like the tips dataset? Why or why not?

*Your answer:*


# Mini Project B: Word Frequency Distribution Analysis (NLP Track)

## Step 1: Build a word frequency table for a longer text

### Demonstration: Zipf's Law preview

A famous finding in linguistics: in most natural language text, word frequency is **highly right-skewed** — a few words (like "the", "a") appear extremely often, and most words appear only once or twice. This is called **Zipf's Law**.


In [ ]:
paragraph = (
    "Data science combines statistics and programming. Statistics helps us understand data. "
    "Programming helps us process data. Machine learning is a part of data science. "
    "Data science uses statistics, programming, and machine learning together. "
    "Understanding statistics is essential for data science."
)

tokens = paragraph.lower().replace('.', '').replace(',', '').split()
tokens_array = np.array(tokens)
vocabulary = np.unique(tokens_array)
counts = [np.sum(tokens_array == w) for w in vocabulary]

word_freq = pd.DataFrame({'word': vocabulary, 'count': counts}).sort_values('count', ascending=False).reset_index(drop=True)
print(word_freq)
print("\nSkewness of word counts:", word_freq['count'].skew())


In [ ]:
plt.figure()
plt.bar(word_freq['word'], word_freq['count'])
plt.xticks(rotation=90)
plt.title("Word Frequency Distribution")
plt.ylabel("Count")
plt.show()


### Student Practice

1. Pick your own longer paragraph (5+ sentences) and build the same word frequency table.
2. Compute the skewness of the word counts. Is it right-skewed (consistent with Zipf's Law)?
3. What percentage of words in your vocabulary appear only once? (This is usually a large percentage — that's Zipf's Law in action.)


In [ ]:
# Write your answer here


## Step 2: Correlation between sentence length and vocabulary diversity

### Demonstration


In [ ]:
sentences = [
    "I like data science.",
    "Machine learning models require lots of clean labeled training data to perform well.",
    "Statistics matters.",
    "Understanding both statistics and programming together is essential for real data science work in industry.",
    "NumPy and pandas are useful tools for data analysis in Python."
]

sentence_stats = []
for s in sentences:
    toks = s.lower().replace('.', '').split()
    sentence_stats.append({
        'sentence': s,
        'length': len(toks),
        'unique_words': len(set(toks))
    })

sentence_df = pd.DataFrame(sentence_stats)
sentence_df['diversity_ratio'] = sentence_df['unique_words'] / sentence_df['length']
sentence_df


In [ ]:
correlation = sentence_df['length'].corr(sentence_df['unique_words'])
print(f"Correlation between sentence length and unique word count: {correlation:.3f}")


### Student Practice

Write 5 sentences of your own, varying in length. Build the same table and compute the correlation between `length` and `unique_words`. Is the correlation strong? Does this match your intuition (longer sentences tend to have more unique words)?


In [ ]:
# Write your answer here


**Reflection:** Why might `diversity_ratio` (unique words / total words) be a more useful metric than raw `unique_words` count when comparing sentences of very different lengths?

*Your answer:*


# Final Self-Check

- [ ] I computed and correctly interpreted skewness (not just mean/std)
- [ ] I visualized a distribution and connected the shape to the skewness number
- [ ] I computed correlation and correctly judged strength (weak/moderate/strong)
- [ ] I ran a t-test and correctly interpreted the p-value (significant or not)
- [ ] I computed a confidence interval and can explain what it means in plain language
- [ ] I completed Mini Project A (image stats) or B (text distribution/correlation)
- [ ] I answered all reflection questions in my own words

## Final Reflection Questions

1. Why is it important to look at data visually (histograms, boxplots) and not just rely on summary numbers like mean and std?

2. If a t-test gives you p = 0.20, what should you conclude, and what should you NOT conclude? (Common misconception to address: "not significant" does not mean "definitely no difference.")

3. How does understanding your data's statistics (distribution shape, correlations) help you make better decisions in the next stage — building an ML model?

4. What's one statistics concept from this lab you still feel unsure about? Be specific.
